# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.9 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = "task314"
TASK_PATH = Path(COMPETITION)/"task314.json"
OUT_DIR = Path.cwd()/"task314_lattice_line_completion_onnx"
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"
SUBMISSION_PATH = Path.cwd()/"submission.zip"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
INPUT_SHAPE = [1, 10, 30, 30]
OUTPUT_SHAPE = [1, 10, 30, 30]

with TASK_PATH.open() as f:
    task = json.load(f)

print({k: len(v) for k, v in task.items()})

{'train': 3, 'test': 1, 'arc-gen': 262}


In [6]:
def encode_grid(grid):
    arr = np.zeros((1, 10, 30, 30), dtype=np.float32)
    h, w = len(grid), len(grid[0])
    assert h <= 30 and w <= 30
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            arr[0, value, r, c] = 1.0
    return arr


def decode_tensor(tensor, h, w):
    return tensor[0, :, :h, :w].argmax(axis=0).astype(int).tolist()


def expected_tensor(grid):
    return encode_grid(grid)


def active_canvas_ok(y, grid):
    h, w = len(grid), len(grid[0])
    outside = y.copy()
    outside[:, :, :h, :w] = 0
    outside_zero = bool(np.all(outside == 0))
    inside_sum = y[:, :, :h, :w].sum(axis=1)
    active_covered = bool(np.all(inside_sum == 1.0))
    return outside_zero, active_covered

In [7]:
def make_lattice_buffers():
    coords = []
    meta = []
    # position index order: local offset (dr, dc), then 3×3 meta-lattice coordinate (R, C)
    for dr in range(2):
        for dc in range(2):
            for R in range(3):
                for C in range(3):
                    coords.append((3 * R + dr, 3 * C + dc))
                    meta.append((dr, dc, R, C))

    pos_idx = torch.tensor([r * 30 + c for r, c in coords], dtype=torch.long)

    pair_p, pair_q, pair_c = [], [], []
    for j, (dr, dc, R, C) in enumerate(meta):
        local_indices = [i for i, m in enumerate(meta) if m[0] == dr and m[1] == dc]
        for p in local_indices:
            _, _, R1, C1 = meta[p]
            for q in local_indices:
                if q < p:
                    continue
                _, _, R2, C2 = meta[q]
                horizontal = (R1 == R2 == R and min(C1, C2) <= C <= max(C1, C2))
                vertical = (C1 == C2 == C and min(R1, R2) <= R <= max(R1, R2))
                if horizontal or vertical:
                    pair_p.append(p)
                    pair_q.append(q)
                    pair_c.append(j)

    pair_p = torch.tensor(pair_p, dtype=torch.long)
    pair_q = torch.tensor(pair_q, dtype=torch.long)

    pair_to_cand = torch.zeros(len(pair_p), 36, dtype=torch.float32)
    for i, j in enumerate(pair_c):
        pair_to_cand[i, j] = 1.0

    pos_to_flat = torch.zeros(36, 900, dtype=torch.float32)
    for i, idx in enumerate(pos_idx):
        pos_to_flat[i, idx] = 1.0

    return pos_idx, pair_p, pair_q, pair_to_cand, pos_to_flat


class Task314LatticeLineCompletion(torch.nn.Module):
    def __init__(self):
        super().__init__()
        pos_idx, pair_p, pair_q, pair_to_cand, pos_to_flat = make_lattice_buffers()
        self.register_buffer("pos_idx", pos_idx)
        self.register_buffer("pair_p", pair_p)
        self.register_buffer("pair_q", pair_q)
        self.register_buffer("pair_to_cand", pair_to_cand)
        self.register_buffer("pos_to_flat", pos_to_flat)

    def forward(self, x):
        b = x.shape[0]
        flat = x.reshape(b, 10, 900)

        # Gather only the 36 non-separator positions of the 3×3 lattice.
        pos = torch.index_select(flat, 2, self.pos_idx)
        markers = pos[:, 2:10, :]

        # For each color and lattice position, detect if it lies between two observed
        # same-color endpoints in the same meta-row or meta-column.
        p = torch.index_select(markers, 2, self.pair_p)
        q = torch.index_select(markers, 2, self.pair_q)
        pair_hits = p * q
        cond = (torch.matmul(pair_hits, self.pair_to_cand) > 0.5).to(x.dtype)

        # Scatter structurally via a fixed incidence matrix into the 30×30 flattened canvas.
        fill8 = torch.matmul(cond, self.pos_to_flat)
        zeros2 = fill8[:, 0:2, :] * 0.0
        fill_full = torch.cat([zeros2, fill8], dim=1)
        fill_any = (fill_full.sum(dim=1, keepdim=True) > 0.5).to(x.dtype)

        y = fill_full * fill_any + flat * (1.0 - fill_any)

        # Required active-canvas masking. Padding is all-zero across all channels and must stay so.
        active_canvas = (flat.sum(dim=1, keepdim=True) > 0.0).to(x.dtype)
        y = y * active_canvas
        return y.reshape(b, 10, 30, 30)


model = Task314LatticeLineCompletion().eval()
print(model)

Task314LatticeLineCompletion()


In [8]:
def validate_torch(split_name, examples):
    ok = 0
    bad = []
    outside_ok = 0
    active_ok = 0
    with torch.no_grad():
        for i, ex in enumerate(examples):
            x = torch.from_numpy(encode_grid(ex["input"]))
            y = model(x).numpy()
            exp = expected_tensor(ex["output"])
            if np.array_equal(y, exp):
                ok += 1
            else:
                bad.append(i)
            oz, ac = active_canvas_ok(y, ex["output"])
            outside_ok += int(oz)
            active_ok += int(ac)
    return {
        "ok": ok,
        "total": len(examples),
        "bad_first10": bad[:10],
        "outside_zero_ok": outside_ok,
        "active_canvas_covered_ok": active_ok,
    }

for split in ["train", "test", "arc-gen"]:
    print(split, validate_torch(split, task[split]))

train {'ok': 3, 'total': 3, 'bad_first10': [], 'outside_zero_ok': 3, 'active_canvas_covered_ok': 3}
test {'ok': 1, 'total': 1, 'bad_first10': [], 'outside_zero_ok': 1, 'active_canvas_covered_ok': 1}
arc-gen {'ok': 262, 'total': 262, 'bad_first10': [], 'outside_zero_ok': 262, 'active_canvas_covered_ok': 262}


In [9]:
dummy = torch.zeros(*INPUT_SHAPE, dtype=torch.float32)
dummy[:, 0, :8, :8] = 1.0

torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
    external_data=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

ops = Counter(node.op_type for node in onnx_model.graph.node)
forbidden = sorted(set(ops) & FORBIDDEN_OPS)
onnx_size = ONNX_PATH.stat().st_size

print("ONNX path:", ONNX_PATH)
print("ONNX size:", onnx_size)
print("ops:", dict(ops))
print("forbidden:", forbidden)
print("function_count:", len(onnx_model.functions))

assert onnx_size < 1_400_000
assert not forbidden
assert len(onnx_model.functions) == 0

/tmp/ipykernel_16/2878347580.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX path: /kaggle/working/task314_lattice_line_completion_onnx/task314.onnx
ONNX size: 166015
ops: {'Constant': 16, 'Reshape': 2, 'Gather': 3, 'Slice': 2, 'Mul': 5, 'MatMul': 2, 'Greater': 3, 'Cast': 3, 'Concat': 1, 'ReduceSum': 2, 'Sub': 1, 'Add': 1}
forbidden: []
function_count: 0


In [10]:
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
input_shape = session.get_inputs()[0].shape
output_shape = session.get_outputs()[0].shape
print("input_shape", input_shape)
print("output_shape", output_shape)
assert list(input_shape) == INPUT_SHAPE
assert list(output_shape) == OUTPUT_SHAPE


def validate_onnx(examples):
    ok = 0
    bad = []
    outside_ok = 0
    active_ok = 0
    for i, ex in enumerate(examples):
        x = encode_grid(ex["input"])
        y = session.run(None, {"input": x})[0]
        exp = expected_tensor(ex["output"])
        if np.array_equal(y, exp):
            ok += 1
        else:
            bad.append(i)
        oz, ac = active_canvas_ok(y, ex["output"])
        outside_ok += int(oz)
        active_ok += int(ac)
    return {
        "ok": ok,
        "total": len(examples),
        "bad_first10": bad[:10],
        "outside_zero_ok": outside_ok,
        "active_canvas_covered_ok": active_ok,
    }

arc_gen = task.get("arc-gen", [])
holdout_n = int(len(arc_gen) * 0.60)
summary = {
    "input_shape": INPUT_SHAPE,
    "output_shape": OUTPUT_SHAPE,
    "onnx_size_bytes": onnx_size,
    "ops": dict(ops),
    "forbidden_ops": forbidden,
    "function_count": len(onnx_model.functions),
    "train": validate_onnx(task["train"]),
    "test": validate_onnx(task["test"]),
    "arc_gen_60pct_holdout": validate_onnx(arc_gen[:holdout_n]),
    "arc_gen_all": validate_onnx(arc_gen),
}

print(json.dumps(summary, indent=2))

assert summary["train"]["ok"] == summary["train"]["total"]
assert summary["test"]["ok"] == summary["test"]["total"]
assert summary["arc_gen_60pct_holdout"]["ok"] == summary["arc_gen_60pct_holdout"]["total"]
assert summary["arc_gen_all"]["ok"] == summary["arc_gen_all"]["total"]

with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)
print("summary path:", SUMMARY_PATH)

input_shape [1, 10, 30, 30]
output_shape [1, 10, 30, 30]
{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 166015,
  "ops": {
    "Constant": 16,
    "Reshape": 2,
    "Gather": 3,
    "Slice": 2,
    "Mul": 5,
    "MatMul": 2,
    "Greater": 3,
    "Cast": 3,
    "Concat": 1,
    "ReduceSum": 2,
    "Sub": 1,
    "Add": 1
  },
  "forbidden_ops": [],
  "function_count": 0,
  "train": {
    "ok": 3,
    "total": 3,
    "bad_first10": [],
    "outside_zero_ok": 3,
    "active_canvas_covered_ok": 3
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first10": [],
    "outside_zero_ok": 1,
    "active_canvas_covered_ok": 1
  },
  "arc_gen_60pct_holdout": {
    "ok": 157,
    "total": 157,
    "bad_first10": [],
    "outside_zero_ok": 157,
    "active_canvas_covered_ok": 157
  },
  "arc_gen_all": {
    "ok": 262,
    "total": 262,
    "bad_first10": [],
    "outside_zero_ok": 262,
    "active_canvas_covered

In [11]:
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("submission:", SUBMISSION_PATH)
print("zip contents:")
with zipfile.ZipFile(SUBMISSION_PATH) as zf:
    for info in zf.infolist():
        print(info.filename, info.file_size)

submission: /kaggle/working/submission.zip
zip contents:
task314.onnx 166015
